# Управление Kafka → MinIO Streaming

Управление streaming jobs из Jupyter.
Один раз запустили — работает в фоне.

In [ ]:
from pyspark.sql import SparkSession

# Получаем существующую сессию (или создаем новую)
spark = SparkSession.builder.getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

## Список активных streaming queries

In [ ]:
# Смотрим все активные streaming jobs
active_streams = spark.streams.active

print(f"Active streaming queries: {len(active_streams)}\n")

for query in active_streams:
    print(f"Query ID: {query.id}")
    print(f"Name: {query.name}")
    print(f"Status: {query.status}")
    print(f"Recent progress:")
    if query.lastProgress:
        print(f"  - Batch ID: {query.lastProgress.get('batchId', 'N/A')}")
        print(f"  - Rows: {query.lastProgress.get('numInputRows', 'N/A')}")
        print(f"  - Duration: {query.lastProgress.get('durationMs', {})}")
    print("-" * 70)

## Мониторинг конкретного query

In [ ]:
# Если у вас есть query из другого notebook
# Найдите его по ID
active_streams = spark.streams.active

if len(active_streams) > 0:
    # Берем первый query
    query = active_streams[0]
    
    print(f"Monitoring query: {query.id}\n")
    
    # Детальная информация
    import json
    if query.lastProgress:
        print(json.dumps(query.lastProgress, indent=2))
    else:
        print("No progress yet")
else:
    print("No active queries")

## Остановка всех queries

In [ ]:
# Остановить ВСЕ streaming queries
# ОСТОРОЖНО: это остановит все активные streams!

confirm = input("Stop ALL streaming queries? (yes/no): ")

if confirm.lower() == 'yes':
    for query in spark.streams.active:
        print(f"Stopping query {query.id}...")
        query.stop()
    print("All queries stopped")
else:
    print("Cancelled")

## Остановка конкретного query по ID

In [ ]:
# Укажите ID query для остановки
query_id_to_stop = "ваш-query-id"  # Возьмите из списка выше

for query in spark.streams.active:
    if query.id == query_id_to_stop:
        print(f"Stopping query {query.id}...")
        query.stop()
        print("Stopped")
        break
else:
    print(f"Query {query_id_to_stop} not found")

## Статистика по записанным файлам

In [ ]:
import s3fs

s3 = s3fs.S3FileSystem(
    key='minioadmin',
    secret='minioadmin',
    client_kwargs={'endpoint_url': 'http://minio:9000'}
)

# Проверяем файлы в MinIO
bucket = "datalake"
path = "topics-streaming/order-events"

files = s3.glob(f'{bucket}/{path}/**/*.parquet')

print(f"Total files: {len(files)}")

if files:
    total_size_mb = sum([s3.size(f) for f in files]) / (1024 * 1024)
    print(f"Total size: {total_size_mb:.2f} MB")
    print(f"Average file size: {total_size_mb / len(files):.2f} MB")
    
    print("\nRecent files:")
    for f in sorted(files)[-10:]:
        size_mb = s3.size(f) / (1024 * 1024)
        print(f"  {f.split('/')[-1]}: {size_mb:.2f} MB")
else:
    print("No files yet")

## Сравнение: Kafka Connect vs Spark Streaming

In [ ]:
import s3fs

s3 = s3fs.S3FileSystem(
    key='minioadmin',
    secret='minioadmin',
    client_kwargs={'endpoint_url': 'http://minio:9000'}
)

# Kafka Connect files
connect_files = s3.glob('datalake/topics/order-events/**/*.parquet')
connect_size = sum([s3.size(f) for f in connect_files]) / (1024 * 1024)

# Spark Streaming files
streaming_files = s3.glob('datalake/topics-streaming/order-events/**/*.parquet')
streaming_size = sum([s3.size(f) for f in streaming_files]) / (1024 * 1024) if streaming_files else 0

print("="*70)
print("KAFKA CONNECT:")
print(f"  Files: {len(connect_files)}")
print(f"  Total size: {connect_size:.2f} MB")
print(f"  Avg file size: {connect_size / len(connect_files):.2f} MB" if connect_files else "N/A")

print("\nSPARK STREAMING:")
print(f"  Files: {len(streaming_files)}")
print(f"  Total size: {streaming_size:.2f} MB")
print(f"  Avg file size: {streaming_size / len(streaming_files):.2f} MB" if streaming_files else "N/A")

if connect_files and streaming_files:
    print("\nIMPROVEMENT:")
    print(f"  File reduction: {len(connect_files)} → {len(streaming_files)} ({len(connect_files)/len(streaming_files):.1f}x)")
    print(f"  Avg file size increase: {connect_size/len(connect_files):.2f} MB → {streaming_size/len(streaming_files):.2f} MB")

print("="*70)